In [5]:
import pandas as pd
import numpy as np


# ============================================================
# 0. 데이터 불러오기
# ============================================================
# UCI 원본 데이터는 세미콜론(;)으로 구분되어 있으므로 sep=";" 지정
df = pd.read_csv(
    "../data/raw/data.csv",
    sep=";"
)

# 혹시 컬럼명 앞뒤에 공백이 있을 경우 제거
df.columns = df.columns.str.strip()

# 원본 데이터 보존을 위해 복사본 생성
# 이후 한국 적용형 전처리는 df_kr에서만 진행
df_kr = df.copy()

print("원본 shape:", df_kr.shape)
display(df_kr.head())


# ============================================================
# 1. Target → 이진분류
# ============================================================
# 원본 Target은 3개 클래스
# - Dropout  : 중도탈락
# - Graduate : 졸업
# - Enrolled : 재학
#
# 프로젝트에서는 "중도탈락 여부"를 예측하는 것이 목적이므로
# 아래와 같이 이진분류 문제로 변경
#
# 1 = Dropout
# 0 = Graduate + Enrolled

df_kr["Target_binary"] = df_kr["Target"].map({
    "Dropout": 1,
    "Graduate": 0,
    "Enrolled": 0
})

# 기존 3-class Target 제거
df_kr.drop(columns=["Target"], inplace=True)

print("\n[Target 분포]")
print(df_kr["Target_binary"].value_counts())

print("\n[Target 비율]")
print(
    df_kr["Target_binary"]
    .value_counts(normalize=True)
    .round(3)
)


# ============================================================
# 2. Application mode
#    포르투갈 입학전형 → 국가 중립적인 입학경로
# ============================================================
# 포르투갈의 세부 입학전형 코드를 그대로 사용하면
# 한국 대학에 적용하기 어려우므로 보다 일반적인 입학경로로 그룹화
#
# 예:
# General          : 일반 입학
# Special          : 특별 전형
# International    : 외국인
# Adult_Learner    : 성인 학습자
# Transfer_Change  : 편입 / 과정 변경
# Previous_HE      : 기존 고등교육 경험
# Vocational       : 직업·기술교육
# Other_Special    : 기타 특별전형

application_mode_map = {

    # 일반 입학
    1: "General",
    17: "General",
    18: "General",

    # 지역 / 특별 전형
    5: "Special",
    16: "Special",

    # 외국인
    15: "International",

    # 성인 학습자
    39: "Adult_Learner",

    # 편입 / 과정 변경
    42: "Transfer_Change",
    43: "Transfer_Change",
    51: "Transfer_Change",
    57: "Transfer_Change",

    # 기존 고등교육 경험
    7: "Previous_HE",

    # 직업 / 기술교육
    44: "Vocational",
    53: "Vocational",

    # 기타 특별 전형
    2: "Other_Special",
    10: "Other_Special",
    26: "Other_Special",
    27: "Other_Special"
}

# 새로운 입학경로 변수 생성
# 매핑되지 않은 코드는 Other로 처리
df_kr["Admission_pathway"] = (
    df_kr["Application mode"]
    .map(application_mode_map)
    .fillna("Other")
)

# 기존 포르투갈 Application mode 제거
df_kr.drop(columns=["Application mode"], inplace=True)


# ============================================================
# 3. Course
#    개별 학과 → 일반화된 전공 계열
# ============================================================
# UCI 데이터의 Course는 포르투갈 대학의 개별 학과 코드이므로
# 한국에서도 해석 가능한 넓은 전공 계열로 그룹화
#
# ⚠️ 아래 grouping은 한국의 공식 학과 분류가 아니라
# 국가 종속성을 낮추기 위한 프로젝트용 분석 분류

course_map = {

    # 농생명 / 자연
    33: "Agriculture_Natural",
    9130: "Agriculture_Natural",
    9085: "Agriculture_Natural",

    # 디자인 / 예술
    171: "Arts_Design",
    9070: "Arts_Design",

    # 사회 / 교육
    8014: "Social_Education",
    9238: "Social_Education",
    9773: "Social_Education",
    9853: "Social_Education",

    # 경영 / 관광 / 서비스
    9003: "Business_Service",
    9254: "Business_Service",
    9500: "Business_Service",

    # 보건
    9119: "Health",
    9556: "Health",
    9670: "Health",

    # 공학 / IT
    9991: "Engineering_IT"
}

# 전공 계열 변수 생성
df_kr["Course_group"] = (
    df_kr["Course"]
    .map(course_map)
    .fillna("Other")
)

# 기존 Course 코드 제거
df_kr.drop(columns=["Course"], inplace=True)


# ============================================================
# 4. Previous qualification
#    포르투갈 교육제도 → 일반적인 이전 학력
# ============================================================
# 포르투갈의 세부 학력 코드를 그대로 사용하는 대신
# 국가 간 비교가 가능한 상위 학력 개념으로 단순화
#
# Secondary        : 중등교육
# Higher_Education : 고등교육 경험
# Below_Secondary  : 중등교육 미만
# Vocational       : 직업교육

previous_qualification_map = {

    # 중등교육
    1: "Secondary",

    # 고등교육
    2: "Higher_Education",
    3: "Higher_Education",
    4: "Higher_Education",
    5: "Higher_Education",
    6: "Higher_Education",

    # 중등교육 수준
    9: "Secondary",
    10: "Secondary",
    12: "Secondary",
    14: "Secondary",
    15: "Secondary",

    # 중등교육 미만
    19: "Below_Secondary",
    38: "Below_Secondary",

    # 직업 / 기술교육
    39: "Vocational",
    40: "Vocational",
    42: "Vocational",
    43: "Vocational"
}

df_kr["Previous_qualification_group"] = (
    df_kr["Previous qualification"]
    .map(previous_qualification_map)
    .fillna("Other")
)

# 기존 세부 학력 코드 제거
df_kr.drop(columns=["Previous qualification"], inplace=True)


# ============================================================
# 5. 부모 학력
#    세부 교육과정 → 학력 수준
# ============================================================
# 부모의 학력 코드 역시 포르투갈 교육제도에 종속되어 있으므로
# 크게 고등교육 / 중등교육 / 기초교육 / 기타로 그룹화

def group_parent_qualification(x):

    # 대학 / 고등교육 계열
    higher = [
        2, 3, 4, 5, 6, 40, 41, 42, 43, 44
    ]

    # 중등교육 계열
    secondary = [
        1, 9, 10, 11, 12, 13, 14
    ]

    # 기초교육 계열
    basic = [
        19, 20, 22, 25, 26, 27, 29, 30
    ]

    if x in higher:
        return "Higher_Education"

    elif x in secondary:
        return "Secondary"

    elif x in basic:
        return "Basic"

    else:
        return "Other"


# 어머니 학력 그룹
df_kr["Mother_education_group"] = (
    df_kr["Mother's qualification"]
    .apply(group_parent_qualification)
)

# 아버지 학력 그룹
df_kr["Father_education_group"] = (
    df_kr["Father's qualification"]
    .apply(group_parent_qualification)
)

# 기존 세부 학력 코드 제거
df_kr.drop(
    columns=[
        "Mother's qualification",
        "Father's qualification"
    ],
    inplace=True
)


# ============================================================
# 6. 부모 직업
#    포르투갈 세부 직업코드 → 광범위한 직업군
# ============================================================
# 국가별 직업분류체계가 다르기 때문에
# 너무 세부적으로 해석하지 않고 큰 직업군으로 그룹화
#
# Professional_Managerial : 관리자 / 전문직
# Clerical_Service        : 사무 / 서비스 / 판매
# Agriculture             : 농림
# Technical_Production    : 기술 / 생산
# Elementary              : 단순노무
# Other                   : 기타

def group_occupation(x):

    # 관리자 / 전문직
    if x in [1, 2, 3]:
        return "Professional_Managerial"

    # 사무 / 서비스 / 판매
    elif x in [4, 5]:
        return "Clerical_Service"

    # 농림
    elif x == 6:
        return "Agriculture"

    # 기술 / 생산
    elif x in [7, 8]:
        return "Technical_Production"

    # 단순노무
    elif x == 9:
        return "Elementary"

    # 기타
    else:
        return "Other"


# 부모 직업 그룹 변수 생성
df_kr["Mother_occupation_group"] = (
    df_kr["Mother's occupation"]
    .apply(group_occupation)
)

df_kr["Father_occupation_group"] = (
    df_kr["Father's occupation"]
    .apply(group_occupation)
)

# 기존 세부 직업 코드 제거
df_kr.drop(
    columns=[
        "Mother's occupation",
        "Father's occupation"
    ],
    inplace=True
)


# ============================================================
# 7. Nacionality 제거
# ============================================================
# 세부 국적 정보는 포르투갈 데이터에 종속적이며,
# 원본 데이터에 International 변수도 존재하므로
# 한국 적용형 모델에서는 세부 국적 변수 제거

df_kr.drop(columns=["Nacionality"], inplace=True)


# ============================================================
# 8. 포르투갈 거시경제 변수 제거
# ============================================================
# 아래 변수들은 데이터 수집 당시 포르투갈의 경제 상황을 의미
#
# - Unemployment rate : 실업률
# - Inflation rate    : 물가상승률
# - GDP               : 국내총생산
#
# 한국 대학생에게 그대로 적용하기 어렵기 때문에 제거

macro_columns = [
    "Unemployment rate",
    "Inflation rate",
    "GDP"
]

df_kr.drop(columns=macro_columns, inplace=True)


# ============================================================
# 9. 학업 관련 파생변수 생성
# ============================================================
# 단순 과목 수나 성적뿐 아니라
# 학생의 학업 진행 상태를 더 쉽게 파악할 수 있도록
# 이수율, 미평가율, 성적 변화 등의 파생변수를 생성
#
# np.where()를 사용하여
# enrolled가 0인 경우 0으로 나누는 문제를 방지


# ------------------------------------------------------------
# 9-1. 1학기 이수율
# 승인 과목 수 / 등록 과목 수
# ------------------------------------------------------------
df_kr["Sem1_approval_rate"] = np.where(
    df_kr["Curricular units 1st sem (enrolled)"] > 0,

    df_kr["Curricular units 1st sem (approved)"]
    / df_kr["Curricular units 1st sem (enrolled)"],

    0
)


# ------------------------------------------------------------
# 9-2. 2학기 이수율
# ------------------------------------------------------------
df_kr["Sem2_approval_rate"] = np.where(
    df_kr["Curricular units 2nd sem (enrolled)"] > 0,

    df_kr["Curricular units 2nd sem (approved)"]
    / df_kr["Curricular units 2nd sem (enrolled)"],

    0
)


# ------------------------------------------------------------
# 9-3. 1학기 미평가 비율
# 미평가 과목 수 / 등록 과목 수
# ------------------------------------------------------------
df_kr["Sem1_no_eval_rate"] = np.where(
    df_kr["Curricular units 1st sem (enrolled)"] > 0,

    df_kr["Curricular units 1st sem (without evaluations)"]
    / df_kr["Curricular units 1st sem (enrolled)"],

    0
)


# ------------------------------------------------------------
# 9-4. 2학기 미평가 비율
# ------------------------------------------------------------
df_kr["Sem2_no_eval_rate"] = np.where(
    df_kr["Curricular units 2nd sem (enrolled)"] > 0,

    df_kr["Curricular units 2nd sem (without evaluations)"]
    / df_kr["Curricular units 2nd sem (enrolled)"],

    0
)


# ------------------------------------------------------------
# 9-5. 성적 변화
# 2학기 평균 성적 - 1학기 평균 성적
#
# 양수 → 성적 상승
# 음수 → 성적 하락
# ------------------------------------------------------------
df_kr["Grade_change"] = (
    df_kr["Curricular units 2nd sem (grade)"]
    - df_kr["Curricular units 1st sem (grade)"]
)


# ------------------------------------------------------------
# 9-6. 이수율 변화
# 2학기 이수율 - 1학기 이수율
#
# 양수 → 학업 진행 개선
# 음수 → 학업 진행 악화
# ------------------------------------------------------------
df_kr["Approval_rate_change"] = (
    df_kr["Sem2_approval_rate"]
    - df_kr["Sem1_approval_rate"]
)


# ============================================================
# 10. 전처리 결과 확인
# ============================================================

print("\n==============================")
print("한국 적용형 데이터")
print("==============================")

# 최종 데이터 크기
print("shape:", df_kr.shape)

# 최종 컬럼 확인
print("\n[컬럼]")
for col in df_kr.columns:
    print("-", col)

# 결측치 확인
print("\n[결측치]")
print(
    df_kr
    .isnull()
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

# Target 분포 확인
print("\n[Target]")
print(df_kr["Target_binary"].value_counts())


# ============================================================
# 11. 전처리 데이터 저장
# ============================================================
# 원본 데이터는 data/raw에 보존하고,
# 한국 적용형 전처리 결과는 data/processed에 별도로 저장

df_kr.to_csv(
    "../data/processed/data_kr_processed.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n전처리 데이터 저장 완료!")
print("../data/processed/data_kr_processed.csv")

원본 shape: (4424, 37)


,Marital status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,17,5,171,1,1,122.0,1,19,12,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,15,1,9254,1,1,160.0,1,1,3,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,9070,1,1,122.0,1,37,37,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,17,2,9773,1,1,122.0,1,38,37,...,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,39,1,8014,0,1,100.0,1,37,38,...,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate



[Target 분포]
Target_binary
0    3003
1    1421
Name: count, dtype: int64

[Target 비율]
Target_binary
0    0.679
1    0.321
Name: proportion, dtype: float64

한국 적용형 데이터
shape: (4424, 39)

[컬럼]
- Marital status
- Application order
- Daytime/evening attendance
- Previous qualification (grade)
- Admission grade
- Displaced
- Educational special needs
- Debtor
- Tuition fees up to date
- Gender
- Scholarship holder
- Age at enrollment
- International
- Curricular units 1st sem (credited)
- Curricular units 1st sem (enrolled)
- Curricular units 1st sem (evaluations)
- Curricular units 1st sem (approved)
- Curricular units 1st sem (grade)
- Curricular units 1st sem (without evaluations)
- Curricular units 2nd sem (credited)
- Curricular units 2nd sem (enrolled)
- Curricular units 2nd sem (evaluations)
- Curricular units 2nd sem (approved)
- Curricular units 2nd sem (grade)
- Curricular units 2nd sem (without evaluations)
- Target_binary
- Admission_pathway
- Course_group
- Previous_qualificati